In [ ]:
# Standard library
import os
import re
from pathlib import Path

# Third-party (alphabetical)
import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from openai import OpenAI
from transformers import AutoTokenizer, AutoModel

# Project paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

In [ ]:

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

# Chunking configuration
FIXED_CHUNK_WORDS = 150
FIXED_CHUNK_OVERLAP = 30

# Columns we expect in the clinical notes dataset
REQUIRED_NOTE_COLUMNS = {
    "person_id",
    "admission_id",
    "clinical_note_id",
    "clean_note_text",
    "creation_timestamp",
    "note_subject",
    "note_type",
}

In [4]:
NOTES_PATH =  '../data/raw/clinical_notes.csv'   # update if filename/path differs

notes = pd.read_csv(NOTES_PATH)


print(f"Loaded {len(notes):,} clinical notes")
notes.head()

Loaded 1,602 clinical notes


,ingest_timestamp,clinical_note_id,clean_note_text,creation_timestamp,updt_dt_tm,note_subject,note_type,admission_id,person_id
0,07/01/2026 14:35,17bf845b-88f8-4604-8983-6e74453aada5,Patient Name: Judith Ada Wells\n- Patient ID: ...,07/01/2026 14:05,07/01/2026 14:35,ED Triage,ED,63720303-3c1b-4356-befd-eea5438da62e,28570119-9cdc-4120-98c0-4edb76cf36a3
1,07/01/2026 14:50,e5d9c0a4-299a-425e-abbc-27fabe9cb742,Patient reviewed at 14:20 on 07/01/26 by Nurse...,07/01/2026 14:20,07/01/2026 14:50,ED Triage Follow-Up,ED,63720303-3c1b-4356-befd-eea5438da62e,28570119-9cdc-4120-98c0-4edb76cf36a3
2,07/01/2026 15:15,1a711621-1094-4d0b-9cec-7925438e19cb,"- Patient: Judith Ad a Wells, 39-year-old fema...",07/01/2026 14:45,07/01/2026 15:15,ED CT Head Scan Review,ED,63720303-3c1b-4356-befd-eea5438da62e,28570119-9cdc-4120-98c0-4edb76cf36a3
3,07/01/2026 15:45,bbbb3acb-d58e-414d-a0de-7c29553b5459,"Patient: Judith Ada Wells, 39-yer-old female, ...",07/01/2026 15:15,07/01/2026 15:45,ED Investigations,ED,63720303-3c1b-4356-befd-eea5438da62e,28570119-9cdc-4120-98c0-4edb76cf36a3
4,07/01/2026 17:00,e3ec2bf0-baa8-4287-8698-592f70a4bccf,Patient\nJudith Ada Wells\n\nAge\n39\n\nSex\nF...,07/01/2026 16:30,07/01/2026 17:00,ED Depart Summary,ED Depart Summary,63720303-3c1b-4356-befd-eea5438da62e,28570119-9cdc-4120-98c0-4edb76cf36a3


In [5]:
missing_columns = REQUIRED_NOTE_COLUMNS - set(notes.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

notes = notes.copy()

notes["creation_timestamp"] = pd.to_datetime(
    notes["creation_timestamp"],
    errors="coerce"
)

notes["clean_note_text"] = (
    notes["clean_note_text"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# Remove rows without usable note text
notes = notes.loc[
    notes["clean_note_text"].ne("")
].reset_index(drop=True)

notes["word_count"] = (
    notes["clean_note_text"]
    .str.split()
    .str.len()
)

print(f"Notes:      {len(notes):,}")
print(f"Patients:   {notes['person_id'].nunique():,}")
print(f"Admissions: {notes['admission_id'].nunique():,}")

Notes:      1,602
Patients:   50
Admissions: 69


In [6]:
def create_whole_note_chunks(notes_df: pd.DataFrame) -> pd.DataFrame:
    """
    Create one retrieval chunk per clinical note.
    """

    chunks = notes_df[
        [
            "person_id",
            "admission_id",
            "clinical_note_id",
            "creation_timestamp",
            "note_subject",
            "note_type",
            "clean_note_text",
        ]
    ].copy()

    chunks = chunks.rename(
        columns={"clean_note_text": "chunk_text"}
    )

    chunks["chunk_index"] = 0

    chunks["chunk_id"] = (
        chunks["clinical_note_id"].astype(str)
        + "_whole_0"
    )

    chunks["chunk_strategy"] = "whole_note"

    return chunks


whole_chunks = create_whole_note_chunks(notes)

print(f"Whole-note chunks: {len(whole_chunks):,}")
whole_chunks.head()

Whole-note chunks: 1,602


,person_id,admission_id,clinical_note_id,creation_timestamp,note_subject,note_type,chunk_text,chunk_index,chunk_id,chunk_strategy
0,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,17bf845b-88f8-4604-8983-6e74453aada5,2026-07-01 14:05:00,ED Triage,ED,Patient Name: Judith Ada Wells\n- Patient ID: ...,0,17bf845b-88f8-4604-8983-6e74453aada5_whole_0,whole_note
1,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,e5d9c0a4-299a-425e-abbc-27fabe9cb742,2026-07-01 14:20:00,ED Triage Follow-Up,ED,Patient reviewed at 14:20 on 07/01/26 by Nurse...,0,e5d9c0a4-299a-425e-abbc-27fabe9cb742_whole_0,whole_note
2,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,1a711621-1094-4d0b-9cec-7925438e19cb,2026-07-01 14:45:00,ED CT Head Scan Review,ED,"- Patient: Judith Ad a Wells, 39-year-old fema...",0,1a711621-1094-4d0b-9cec-7925438e19cb_whole_0,whole_note
3,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,bbbb3acb-d58e-414d-a0de-7c29553b5459,2026-07-01 15:15:00,ED Investigations,ED,"Patient: Judith Ada Wells, 39-yer-old female, ...",0,bbbb3acb-d58e-414d-a0de-7c29553b5459_whole_0,whole_note
4,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,e3ec2bf0-baa8-4287-8698-592f70a4bccf,2026-07-01 16:30:00,ED Depart Summary,ED Depart Summary,Patient\nJudith Ada Wells\n\nAge\n39\n\nSex\nF...,0,e3ec2bf0-baa8-4287-8698-592f70a4bccf_whole_0,whole_note


In [7]:
def split_fixed_words(
    text: str,
    chunk_size: int = 150,
    overlap: int = 30,
) -> list[str]:
    """
    Split text into fixed-size word chunks with overlap.
    """

    if chunk_size <= 0:
        raise ValueError("chunk_size must be greater than 0")

    if overlap < 0:
        raise ValueError("overlap cannot be negative")

    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    words = text.split()

    if len(words) <= chunk_size:
        return [text]

    chunks = []
    step = chunk_size - overlap

    for start in range(0, len(words), step):
        end = start + chunk_size
        chunk_words = words[start:end]

        if not chunk_words:
            break

        chunks.append(" ".join(chunk_words))

        if end >= len(words):
            break

    return chunks


def create_fixed_word_chunks(
    notes_df: pd.DataFrame,
    chunk_size: int = 150,
    overlap: int = 30,
) -> pd.DataFrame:
    """
    Create fixed-size word chunks from each clinical note.
    """

    records = []

    for row in notes_df.itertuples(index=False):

        text_chunks = split_fixed_words(
            row.clean_note_text,
            chunk_size=chunk_size,
            overlap=overlap,
        )

        for chunk_index, chunk_text in enumerate(text_chunks):

            records.append(
                {
                    "person_id": row.person_id,
                    "admission_id": row.admission_id,
                    "clinical_note_id": row.clinical_note_id,
                    "creation_timestamp": row.creation_timestamp,
                    "note_subject": row.note_subject,
                    "note_type": row.note_type,
                    "chunk_index": chunk_index,
                    "chunk_id": (
                        f"{row.clinical_note_id}"
                        f"_fixed_{chunk_index}"
                    ),
                    "chunk_strategy": "fixed_words",
                    "chunk_text": chunk_text,
                }
            )

    return pd.DataFrame(records)


fixed_chunks = create_fixed_word_chunks(
    notes,
    chunk_size=FIXED_CHUNK_WORDS,
    overlap=FIXED_CHUNK_OVERLAP,
)

print(f"Fixed-word chunks: {len(fixed_chunks):,}")
fixed_chunks.head()

Fixed-word chunks: 2,298


,person_id,admission_id,clinical_note_id,creation_timestamp,note_subject,note_type,chunk_index,chunk_id,chunk_strategy,chunk_text
0,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,17bf845b-88f8-4604-8983-6e74453aada5,2026-07-01 14:05:00,ED Triage,ED,0,17bf845b-88f8-4604-8983-6e74453aada5_fixed_0,fixed_words,Patient Name: Judith Ada Wells\n- Patient ID: ...
1,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,e5d9c0a4-299a-425e-abbc-27fabe9cb742,2026-07-01 14:20:00,ED Triage Follow-Up,ED,0,e5d9c0a4-299a-425e-abbc-27fabe9cb742_fixed_0,fixed_words,Patient reviewed at 14:20 on 07/01/26 by Nurse...
2,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,1a711621-1094-4d0b-9cec-7925438e19cb,2026-07-01 14:45:00,ED CT Head Scan Review,ED,0,1a711621-1094-4d0b-9cec-7925438e19cb_fixed_0,fixed_words,"- Patient: Judith Ad a Wells, 39-year-old fema..."
3,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,bbbb3acb-d58e-414d-a0de-7c29553b5459,2026-07-01 15:15:00,ED Investigations,ED,0,bbbb3acb-d58e-414d-a0de-7c29553b5459_fixed_0,fixed_words,"Patient: Judith Ada Wells, 39-yer-old female, ..."
4,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,e3ec2bf0-baa8-4287-8698-592f70a4bccf,2026-07-01 16:30:00,ED Depart Summary,ED Depart Summary,0,e3ec2bf0-baa8-4287-8698-592f70a4bccf_fixed_0,fixed_words,Patient Judith Ada Wells Age 39 Sex Female NHS...


In [20]:
SECTION_HEADINGS = [
    "Presenting Complaint",
    "History of Presenting Illness",
    "History of Present Illness",
    "HPI",
    "Review of Systems",
    "Past Medical History",
    "PMH",
    "Medications",
    "Medication",
    "Allergies",
    "Social History",
    "Family History",
    "On Examination",
    "Examination",
    "Observations",
    "Investigations",
    "Test Results",
    "Results",
    "Assessment",
    "Impression",
    "Diagnosis",
    "Treatment",
    "Plan",
]

SECTION_PATTERN = re.compile(
    rf"(?im)^(?:{'|'.join(map(re.escape, SECTION_HEADINGS))})\s*:?\s*$"
)


def is_heading_only(text: str) -> bool:
    """
    Return True if the chunk contains only a recognized section heading
    and no clinical content.
    """
    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    if len(lines) != 1:
        return False

    return bool(SECTION_PATTERN.fullmatch(lines[0]))


def split_by_sections(text: str) -> list[str]:
    """
    Split a clinical note using known section headings.

    - Keeps section heading together with its content.
    - Drops empty sections that contain only a heading.
    - Falls back to the whole note when usable section boundaries
      are not found.
    """

    matches = list(SECTION_PATTERN.finditer(text))

    if len(matches) < 2:
        return [text]

    chunks = []

    # Preserve text before first recognized heading
    prefix = text[:matches[0].start()].strip()

    if prefix:
        chunks.append(prefix)

    for index, match in enumerate(matches):

        start = match.start()

        if index + 1 < len(matches):
            end = matches[index + 1].start()
        else:
            end = len(text)

        section = text[start:end].strip()

        if section and not is_heading_only(section):
            chunks.append(section)

    return chunks


def create_section_chunks(
    notes_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Create section-aware chunks from clinical notes.

    Notes without detectable sections remain whole.
    """

    records = []

    for row in notes_df.itertuples(index=False):

        text_chunks = split_by_sections(
            row.clean_note_text
        )

        for chunk_index, chunk_text in enumerate(text_chunks):

            records.append(
                {
                    "person_id": row.person_id,
                    "admission_id": row.admission_id,
                    "clinical_note_id": row.clinical_note_id,
                    "creation_timestamp": row.creation_timestamp,
                    "note_subject": row.note_subject,
                    "note_type": row.note_type,
                    "chunk_index": chunk_index,
                    "chunk_id": (
                        f"{row.clinical_note_id}"
                        f"_section_{chunk_index}"
                    ),
                    "chunk_strategy": "section",
                    "chunk_text": chunk_text,
                }
            )

    return pd.DataFrame(records)


section_chunks = create_section_chunks(notes)

print(f"Section chunks: {len(section_chunks):,}")
section_chunks.head()

Section chunks: 5,189


Section chunks: 5,189


,person_id,admission_id,clinical_note_id,creation_timestamp,note_subject,note_type,chunk_index,chunk_id,chunk_strategy,chunk_text
0,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,17bf845b-88f8-4604-8983-6e74453aada5,2026-07-01 14:05:00,ED Triage,ED,0,17bf845b-88f8-4604-8983-6e74453aada5_section_0,section,"Patient Name: Judith Ada Wells\n- Patient ID: 28570119-9cdc-4120-98c0-4edb76cf36a3\n- NHS Number: 272733208\n- Date of Birth: 15/05/84 (39 years old)\n- Gender: Female\n- Allergies: NKA\n- Current Medications: No current medications\n\nTriage Details:\n- Date:07/01/26\n- Time: 14:05\n- Triage Category: Category 2 (Urgent - potentially serious condition requiring prompt atention)\n- Chief Complaint: Severe headache after exertion, rated 8/10 in intensity\n- Nurse: Jasmine Freda Murray\n\nInitial Observations:\n- BP: 160/90 mmHg\n- HR: 88 bpm\n\nED Diagnosis:\n- RCVS\n\nNext Steps:\n- Decision to perform neurological assessment\n- Urgent investigations planned, including CT head\n- Admitting Consultant: Dr. Kevin Richard Martin\nNurse Jasmine Freda Murray \nNMC number: 20F4626L"
1,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,e5d9c0a4-299a-425e-abbc-27fabe9cb742,2026-07-01 14:20:00,ED Triage Follow-Up,ED,0,e5d9c0a4-299a-425e-abbc-27fabe9cb742_section_0,section,Patient reviewed at 14:20 on 07/01/26 by Nurse Chukwuebuka Okafor. Patient presented with a severe headache rated 8/10 in intensity. BP measured at 160/90 mmHg. HR recorded at 88 bpm. Brief neurological examination performed; no abnormalities detected. Decision made to proceed with CT head scan to rule out intracrranial causes for headache.\nNurse Chukwuebuka Okafor \nNMC number: 18D6896L
2,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,1a711621-1094-4d0b-9cec-7925438e19cb,2026-07-01 14:45:00,ED CT Head Scan Review,ED,0,1a711621-1094-4d0b-9cec-7925438e19cb_section_0,section,"- Patient: Judith Ad a Wells, 39-year-old female, DOB: 15/05/84, NHS Number: 272733208.\n - Date/Time: 07/01/26, 14:45.\n - Staff involved: Nurse Jasmine Freda Murray.\n - Chief Complaint: Severe headache after exertion, rated 8/10 in intensity.\n - Initial Observations: BP of 160/90 mmHg and HR of 88 bpm recorded during triage.\n - Event details: CT head scan performed to rule out intracranial causes for the headache. Findings: No evidence of intracranial haemorrhage or mass lesion.\n - Next steps: No immediate medication changes. Blood tests arranged for further investigation.\nNurse Jasmine Freda Murray \nNMC number: 20F4626L"
3,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,bbbb3acb-d58e-414d-a0de-7c29553b5459,2026-07-01 15:15:00,ED Investigations,ED,0,bbbb3acb-d58e-414d-a0de-7c29553b5459_section_0,section,"Patient: Judith Ada Wells, 39-yer-old female, presenting with severe headache rated 8/10 in intensity after exertion. Current triage category: Category 2 (Urgent - potentially serious condition requiring prompt attention). Diagnosis: RCVS. Initial triage performed by Nurse Chukwuebuka Okafor at 14:20 recorded BP at 160/90 mmHg and HR at 88 bpm, with no abnormalities on a brief neurological examination. A CT head scan was performed at 14:45 by Nurse Jasmine Freda Murray, confirming no evidence of intracranial hemorrhage or mass lesion. At 15:15, blood samples were collected for FBC, renal panel, LFTs, and inflammatory markers. No immediate medication changes or additions were made at this time. Awaiting test results to guide further management plan. Care provided by Nurse Jasmine Freda M urray.\nNurse Jasmine Freda Murray \nNMC number: 20F4626L"
4,28570119-9cdc-4120-98c0-4edb76cf36a3,63720303-3c1b-4356-befd-eea5438da62e,e3ec2bf0-baa8-4287-8698-592f70a4bccf,2026-07-01 16:30:00,ED Depart Summary,ED Depart Summary,0,e3ec2bf0-baa8-4287-8698-592f70a4bccf_section_0,section,"Patient\nJudith Ada Wells\n\nAge\n39\n\nSex\nFemale\n\nNHS No.\n272733208\n\nDate/Time\n07/01/26 16:30\n\nSeen By\nDr. Vic

In [21]:
chunk_summary = pd.DataFrame(
    {
        "strategy": [
            "Whole note",
            "Fixed words",
            "Section aware",
        ],
        "num_chunks": [
            len(whole_chunks),
            len(fixed_chunks),
            len(section_chunks),
        ],
        "avg_words_per_chunk": [
            whole_chunks["chunk_text"]
            .str.split()
            .str.len()
            .mean(),

            fixed_chunks["chunk_text"]
            .str.split()
            .str.len()
            .mean(),

            section_chunks["chunk_text"]
            .str.split()
            .str.len()
            .mean(),
        ],
    }
)

chunk_summary

,strategy,num_chunks,avg_words_per_chunk
0,Whole note,1602,129.997503
1,Fixed words,2298,99.711053
2,Section aware,5189,40.124880


In [22]:
section_counts = (
    section_chunks
    .groupby("clinical_note_id")
    .size()
)

print(
    "Notes split into multiple sections:",
    (section_counts > 1).sum()
)

print(
    "Notes left whole:",
    (section_counts == 1).sum()
)

print(
    "Percentage split:",
    round(
        (section_counts > 1).mean() * 100,
        2,
    ),
    "%"
)

Notes split into multiple sections: 569
Notes left whole: 1033
Percentage split: 35.52 %


In [23]:
example_note_id = (
    notes
    .sort_values("word_count", ascending=False)
    .iloc[0]["clinical_note_id"]
)

example_note = notes.loc[
    notes["clinical_note_id"] == example_note_id,
    [
        "clinical_note_id",
        "note_subject",
        "word_count",
        "clean_note_text",
    ],
]

example_note


print("WHOLE NOTE")
print("=" * 80)

display(
    whole_chunks.loc[
        whole_chunks["clinical_note_id"] == example_note_id,
        ["chunk_index", "chunk_text"],
    ]
)

print("\nFIXED WORD")
print("=" * 80)

display(
    fixed_chunks.loc[
        fixed_chunks["clinical_note_id"] == example_note_id,
        ["chunk_index", "chunk_text"],
    ]
)

print("\nSECTION")
print("=" * 80)

display(
    section_chunks.loc[
        section_chunks["clinical_note_id"] == example_note_id,
        ["chunk_index", "chunk_text"],
    ]
)

WHOLE NOTE


WHOLE NOTE


,chunk_index,chunk_text
93,0,"Clerking Doctor\nDr. Kelly Nicola Hayward (SpR)\n\nPresenting Complaint\nAcute confusion following minor fall\n\nHistory of Presenting Complaint\n- Pt reports tripping over a loose carpet edge at home earlier in the day (02/01/26)\n- No LOC reported but recalls feeling dazed afterward\n- Developed pprogressive confusion over the following hours\n- Unable to remember recent events clearly\n- Denies headache, visual changes, N&V, or limb weakness\n- No reported CP, palpitations, or SOB\n\nReview of Systems\n- CNS: Mild confusion, denies heaadche, no visual disturbances, no speech difficulti es, no limb weakness, denies seizures\n- CVS: Denies chest pain, palpitations, or syncope\n- Resp: No breathlessness or cough\n- GIT: No abdominal pain, nausea, vomiting, or changes in bowel habit\n- Renal: No dysuria or haematuria\n- MSK: Reports mild right knee pain following the fall\n- Endo: No polyuria, polydipsia, or heat/cold intolerance\n- Other: No recent fever or weight changes\n\nPast Medical History\n- HTN\n- Mild osteoarthritis\n\nMedications\n- No regular medications\n- IV paracetamol 1g QID prescribed during ED stay to manage pain and prevent discomfort\n\nAllergies\nNone\n\nSocial History\n- Lives alone in a ground-floor flat\n- Independent with activities of daily living\n- Retired accountant\n- Non-smoker\n- Drinks alcohol occasionally, approximately 6 units per week\n- No recreational drug use\n\nFamily History\n- Father: Deceased, history of MI at age 67\n- Motherr: Deceased, history of HTN and stroke\n- No known family history of neurodegenerative or psychiatric disorders\n\nOn Examination\n- Alert but mildly confused, GCS 14/15 (disoriented to time)\n- Appears dehydrated with dry mucous membranes\n- No obvious signs of head trauma or external injuries\n- Neurological examination:\n - Cranial nerves: Intact, pupils equal and reactive to light bilaterally\n - Motor: Normal power (5/5) in all limbs\n - Sensory: No deficits detected in light touch, pinprick, or vibratkion sensation\n - Reflexes: Normal and symmetrical in all limbs, plantar reflexes downgoing bilaterally\n - Coordination: No dysmetria or intention tremor on finger-nose testing\n - Gait: Unsteady, requires assistance to walk; no gross abnormalities in stance or heel-to-toe walking\n- Cardiovascular: Heart sounds dual, no murmurs, no peripheral oedema\n- Respiratory: Equal air entry bilaterally, no added sounds\n- Abdomen: Soft, non-tender, no organomegaly\n- No meningeal signs\n\nObservations\nHR 88\nBP 142/86\nRR 16\nTemp 36.8\nSpO2 98% on room air\n\nInvestigations\n- CT head (02/01/26): DSubdural hygroma noted, no MLS\n- No further imaging performed or currently planned\n\nTest Results\n- FB: Normal\n- U&E: Mild hyponatremia (Na 132 mmol/L)\n- CRP: Normal\n- LFTs: Normal\n- Coagulation profile: Normal\n- Blood glucose: Normal\n- Pending: None\n\nImpression\nAcute confusion secondary to subdural hygroma and mild hypoNa, likely exacerbated by dehydration\n\nPlan\n- Commence IV 0.9% sodium chloride, 1L over 8 hours, to correct dehydration and mild hyponatremia\n- Prescribe IV paracetamol 1g QID to mnaage pain and prevent discomfort\n- Continue monitoring neurological status with GCS assessments every 4 hours\n- Repeat U&E in 24 hours to assess resposne to fluid resuscitation\n- Ensure adequate oral hydration once IV fluids are discontinued\n- Monitor for any progression of symptoms or development of focal neurological deficits\n- Liaise with neurology team for ongoing care and management\n- Document any further findings during the patientâ€™s stay\n\nAdmitting Consultant\nDr. Ismel Siddique (Neurology Consultant)\n\n\nDr. Kelly Nicola Hayward (Specialty Registrar) \nGMC number: 1746274"


WHOLE NOTE


,chunk_index,chunk_text
93,0,"Clerking Doctor\nDr. Kelly Nicola Hayward (SpR)\n\nPresenting Complaint\nAcute confusion following minor fall\n\nHistory of Presenting Complaint\n- Pt reports tripping over a loose carpet edge at home earlier in the day (02/01/26)\n- No LOC reported but recalls feeling dazed afterward\n- Developed pprogressive confusion over the following hours\n- Unable to remember recent events clearly\n- Denies headache, visual changes, N&V, or limb weakness\n- No reported CP, palpitations, or SOB\n\nReview of Systems\n- CNS: Mild confusion, denies heaadche, no visual disturbances, no speech difficulti es, no limb weakness, denies seizures\n- CVS: Denies chest pain, palpitations, or syncope\n- Resp: No breathlessness or cough\n- GIT: No abdominal pain, nausea, vomiting, or changes in bowel habit\n- Renal: No dysuria or haematuria\n- MSK: Reports mild right knee pain following the fall\n- Endo: No polyuria, polydipsia, or heat/cold intolerance\n- Other: No recent fever or weight changes\n\nPast Medical History\n- HTN\n- Mild osteoarthritis\n\nMedications\n- No regular medications\n- IV paracetamol 1g QID prescribed during ED stay to manage pain and prevent discomfort\n\nAllergies\nNone\n\nSocial History\n- Lives alone in a ground-floor flat\n- Independent with activities of daily living\n- Retired accountant\n- Non-smoker\n- Drinks alcohol occasionally, approximately 6 units per week\n- No recreational drug use\n\nFamily History\n- Father: Deceased, history of MI at age 67\n- Motherr: Deceased, history of HTN and stroke\n- No known family history of neurodegenerative or psychiatric disorders\n\nOn Examination\n- Alert but mildly confused, GCS 14/15 (disoriented to time)\n- Appears dehydrated with dry mucous membranes\n- No obvious signs of head trauma or external injuries\n- Neurological examination:\n - Cranial nerves: Intact, pupils equal and reactive to light bilaterally\n - Motor: Normal power (5/5) in all limbs\n - Sensory: No deficits detected in light touch, pinprick, or vibratkion sensation\n - Reflexes: Normal and symmetrical in all limbs, plantar reflexes downgoing bilaterally\n - Coordination: No dysmetria or intention tremor on finger-nose testing\n - Gait: Unsteady, requires assistance to walk; no gross abnormalities in stance or heel-to-toe walking\n- Cardiovascular: Heart sounds dual, no murmurs, no peripheral oedema\n- Respiratory: Equal air entry bilaterally, no added sounds\n- Abdomen: Soft, non-tender, no organomegaly\n- No meningeal signs\n\nObservations\nHR 88\nBP 142/86\nRR 16\nTemp 36.8\nSpO2 98% on room air\n\nInvestigations\n- CT head (02/01/26): DSubdural hygroma noted, no MLS\n- No further imaging performed or currently planned\n\nTest Results\n- FB: Normal\n- U&E: Mild hyponatremia (Na 132 mmol/L)\n- CRP: Normal\n- LFTs: Normal\n- Coagulation profile: Normal\n- Blood glucose: Normal\n- Pending: None\n\nImpression\nAcute confusion secondary to subdural hygroma and mild hypoNa, likely exacerbated by dehydration\n\nPlan\n- Commence IV 0.9% sodium chloride, 1L over 8 hours, to correct dehydration and mild hyponatremia\n- Prescribe IV paracetamol 1g QID to mnaage pain and prevent discomfort\n- Continue monitoring neurological status with GCS assessments every 4 hours\n- Repeat U&E in 24 hours to assess resposne to fluid resuscitation\n- Ensure adequate oral hydration once IV fluids are discontinued\n- Monitor for any progression of symptoms or development of focal neurological deficits\n- Liaise with neurology team for ongoing care and management\n- Document any further findings during the patientâ€™s stay\n\nAdmitting Consultant\nDr. Ismel Siddique (Neurology Consultant)\n\n\nDr. Kelly Nicola Hayward (Specialty Registrar) \nGMC number: 1746274"



FIXED WORD


,chunk_index,chunk_text
130,0,"Clerking Doctor Dr. Kelly Nicola Hayward (SpR) Presenting Complaint Acute confusion following minor fall History of Presenting Complaint - Pt reports tripping over a loose carpet edge at home earlier in the day (02/01/26) - No LOC reported but recalls feeling dazed afterward - Developed pprogressive confusion over the following hours - Unable to remember recent events clearly - Denies headache, visual changes, N&V, or limb weakness - No reported CP, palpitations, or SOB Review of Systems - CNS: Mild confusion, denies heaadche, no visual disturbances, no speech difficulti es, no limb weakness, denies seizures - CVS: Denies chest pain, palpitations, or syncope - Resp: No breathlessness or cough - GIT: No abdominal pain, nausea, vomiting, or changes in bowel habit - Renal: No dysuria or haematuria - MSK: Reports mild right knee pain following the fall - Endo: No polyuria, polydipsia, or heat/cold intolerance - Other: No recent fever"
131,1,"habit - Renal: No dysuria or haematuria - MSK: Reports mild right knee pain following the fall - Endo: No polyuria, polydipsia, or heat/cold intolerance - Other: No recent fever or weight changes Past Medical History - HTN - Mild osteoarthritis Medications - No regular medications - IV paracetamol 1g QID prescribed during ED stay to manage pain and prevent discomfort Allergies None Social History - Lives alone in a ground-floor flat - Independent with activities of daily living - Retired accountant - Non-smoker - Drinks alcohol occasionally, approximately 6 units per week - No recreational drug use Family History - Father: Deceased, history of MI at age 67 - Motherr: Deceased, history of HTN and stroke - No known family history of neurodegenerative or psychiatric disorders On Examination - Alert but mildly confused, GCS 14/15 (disoriented to time) - Appears dehydrated with dry mucous membranes - No obvious signs"
132,2,"family history of neurodegenerative or psychiatric disorders On Examination - Alert but mildly confused, GCS 14/15 (disoriented to time) - Appears dehydrated with dry mucous membranes - No obvious signs of head trauma or external injuries - Neurological examination: - Cranial nerves: Intact, pupils equal and reactive to light bilaterally - Motor: Normal power (5/5) in all limbs - Sensory: No deficits detected in light touch, pinprick, or vibratkion sensation - Reflexes: Normal and symmetrical in all limbs, plantar reflexes downgoing bilaterally - Coordination: No dysmetria or intention tremor on finger-nose testing - Gait: Unsteady, requires assistance to walk; no gross abnormalities in stance or heel-to-toe walking - Cardiovascular: Heart sounds dual, no murmurs, no peripheral oedema - Respiratory: Equal air entry bilaterally, no added sounds - Abdomen: Soft, non-tender, no organomegaly - No meningeal signs Observations HR 88 BP 142/86 RR 16 Temp 36.8 SpO2 98% on room air"
133,3,"air entry bilaterally, no added sounds - Abdomen: Soft, non-tender, no organomegaly - No meningeal signs Observations HR 88 BP 142/86 RR 16 Temp 36.8 SpO2 98% on room air Investigations - CT head (02/01/26): DSubdural hygroma noted, no MLS - No further imaging performed or currently planned Test Results - FB: Normal - U&E: Mild hyponatremia (Na 132 mmol/L) - CRP: Normal - LFTs: Normal - Coagulation profile: Normal - Blood glucose: Normal - Pending: None Impression Acute confusion secondary to subdural hygroma and mild hypoNa, likely exacerbated by dehydration Plan - Commence IV 0.9% sodium chloride, 1L over 8 hours, to correct dehydration and mild hyponatremia - Prescribe IV paracetamol 1g QID to mnaage pain and prevent discomfort - Continue monitoring neurological status with GCS assessments every 4 hours - Repeat U&E in 24 hours to assess resposne to fluid resuscitation - Ensure adequate oral hydration once IV"
134,4,- Continue monitoring neurological status with GCS assessments every 4 hours - Repeat U&E in 24 hours to assess resposne to fluid resuscitation - Ensure a


SECTION


,chunk_index,chunk_text
336,0,Clerking Doctor\nDr. Kelly Nicola Hayward (SpR)
337,1,"Presenting Complaint\nAcute confusion following minor fall\n\nHistory of Presenting Complaint\n- Pt reports tripping over a loose carpet edge at home earlier in the day (02/01/26)\n- No LOC reported but recalls feeling dazed afterward\n- Developed pprogressive confusion over the following hours\n- Unable to remember recent events clearly\n- Denies headache, visual changes, N&V, or limb weakness\n- No reported CP, palpitations, or SOB"
338,2,"Review of Systems\n- CNS: Mild confusion, denies heaadche, no visual disturbances, no speech difficulti es, no limb weakness, denies seizures\n- CVS: Denies chest pain, palpitations, or syncope\n- Resp: No breathlessness or cough\n- GIT: No abdominal pain, nausea, vomiting, or changes in bowel habit\n- Renal: No dysuria or haematuria\n- MSK: Reports mild right knee pain following the fall\n- Endo: No polyuria, polydipsia, or heat/cold intolerance\n- Other: No recent fever or weight changes"
339,3,Past Medical History\n- HTN\n- Mild osteoarthritis
340,4,Medications\n- No regular medications\n- IV paracetamol 1g QID prescribed during ED stay to manage pain and prevent discomfort
341,5,Allergies\nNone
342,6,"Social History\n- Lives alone in a ground-floor flat\n- Independent with activities of daily living\n- Retired accountant\n- Non-smoker\n- Drinks alcohol occasionally, approximately 6 units per week\n- No recreational drug use"
343,7,"Family History\n- Father: Deceased, history of MI at age 67\n- Motherr: Deceased, history of HTN and stroke\n- No known family history of neurodegenerative or psychiatric disorders"
344,8,"On Examination\n- Alert but mildly confused, GCS 14/15 (disoriented to time)\n- Appears dehydrated with dry mucous membranes\n- No obvious signs of head trauma or external injuries\n- Neurological examination:\n - Cranial nerves: Intact, pupils equal and reactive to light bilaterally\n - Motor: Normal power (5/5) in all limbs\n - Sensory: No deficits detected in light touch, pinprick, or vibratkion sensation\n - Reflexes: Normal and symmetrical in all limbs, plantar reflexes downgoing bilaterally\n - Coordination: No dysmetria or intention tremor on finger-nose testing\n - Gait: Unsteady, requires assistance to walk; no gross abnormalities in stance or heel-to-toe walking\n- Cardiovascular: Heart sounds dual, no murmurs, no peripheral oedema\n- Respiratory: Equal air entry bilaterally, no added sounds\n- Abdomen: Soft, non-tender, no organomegaly\n- No meningeal signs"
345,9,Observations\nHR 88\nBP 142/86\nRR 16\nTemp 36.8\nSpO2 98% on room air


In [25]:
chunk_summary

,strategy,num_chunks,avg_words_per_chunk
0,Whole note,1602,129.997503
1,Fixed words,2298,99.711053
2,Section aware,5189,40.124880


In [27]:
section_counts = (
    section_chunks
    .groupby("clinical_note_id")
    .size()
)

print("Notes split into multiple sections:", (section_counts > 1).sum())
print("Notes left whole:", (section_counts == 1).sum())
print(
    "Percentage split:",
    round((section_counts > 1).mean() * 100, 2),
    "%"
)

Notes split into multiple sections: 569
Notes left whole: 1033
Percentage split: 35.52 %


In [28]:
section_chunks = section_chunks.copy()

section_chunks["chunk_word_count"] = (
    section_chunks["chunk_text"]
    .str.split()
    .str.len()
)

section_chunks["chunk_word_count"].describe(
    percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
)

count    5189.000000
mean       40.124880
std        42.022617
min         1.000000
10%         7.000000
25%        11.000000
50%        29.000000
75%        52.000000
90%        88.000000
95%       130.000000
max       385.000000
Name: chunk_word_count, dtype: float64

In [29]:
for threshold in [5, 10, 20]:
    count = (section_chunks["chunk_word_count"] < threshold).sum()
    percentage = count / len(section_chunks) * 100

    print(
        f"Chunks < {threshold} words: "
        f"{count:,} ({percentage:.1f}%)"
    )

Chunks < 5 words: 321 (6.2%)
Chunks < 10 words: 1,078 (20.8%)
Chunks < 20 words: 2,190 (42.2%)


In [30]:
small_chunks = (
    section_chunks.loc[
        section_chunks["chunk_word_count"] < 10,
        [
            "clinical_note_id",
            "note_subject",
            "chunk_index",
            "chunk_word_count",
            "chunk_text",
        ],
    ]
    .sort_values("chunk_word_count")
)

print(f"Number of chunks <10 words: {len(small_chunks)}")

pd.set_option("display.max_colwidth", None)

small_chunks.head(30)

Number of chunks <10 words: 1078


Number of chunks <10 words: 1078


,clinical_note_id,note_subject,chunk_index,chunk_word_count,chunk_text
4370,a62ff755-7c96-4473-ae8b-f7b468cfb771,Dietary Assessment and Plan,0,1,#NAME?
3246,4bf1b7ee-5bdd-40f6-93b3-f576c7158c66,Dietitian Review,0,1,#NAME?
812,0940fa5e-f355-4cea-8e17-61465ea4ff7c,Physio-led education session,0,1,#NAME?
4834,5f6eacde-a2b9-49dd-ac1d-27ac58b0c2de,Dietary review post-op,0,1,#NAME?
2526,170c8e71-5d7f-4bf7-a74f-41aa2a2d800c,Dietitian review for post-op nutrition,0,1,#NAME?
1682,d8df8be7-3622-4782-857f-8d389ddad694,Dietitian Review,0,1,#NAME?
962,661de17e-a4a7-4549-857a-b78e5ce2decf,Dietitian review for post-op nutrition,0,1,#NAME?
3012,6bc4db9a-6d69-460e-a8d8-1657f6c79dc9,ED Depart Summary,4,2,Allergies\nPollen
1071,13ee5295-939b-4ebf-add9-f8c971f66a42,PMWR Ortho Reg,2,2,Investigations\nNone
3242,10ffbd24-1f2e-4303-8daf-873967db1ea3,Neuro AMWR,2,2,Investigations\nNone


In [31]:
investigation_only = section_chunks.loc[
    section_chunks["chunk_text"].str.strip().eq("Investigations"),
    [
        "clinical_note_id",
        "note_subject",
        "chunk_index",
    ],
]

print(
    f"Standalone 'Investigations' chunks: "
    f"{len(investigation_only)}"
)

investigation_only.head(10)

Standalone 'Investigations' chunks: 0


Standalone 'Investigations' chunks: 0


,clinical_note_id,note_subject,chunk_index


In [33]:
small_chunk_text_counts = (
    section_chunks.loc[
        section_chunks["chunk_word_count"] < 10,
        "chunk_text"
    ]
    .str.strip()
    .value_counts()
    .head(30)
)

small_chunk_text_counts

chunk_text
Investigations\nNone                                                   62
Past Medical History\nNil                                              27
Clinician Leading Ward Round\nDr. Sade Olowoyeye (SpR)                 26
Medications\nNil                                                       25
Medications\nNone                                                      21
Clinician Leading Ward Round\nDr. Marcus John Whitehead (SpR)          20
Allergies\nPollen                                                      19
Clinician Leading Ward Round\nDr. Tao Tang (SpR)                       18
Clinician Leading Ward Round\nDr. Stuart Thomas Payne (SpR)            16
Allergies\nNil                                                         14
Clinician Leading Ward Round\nDr. Brenda Veronica Miles (SpR)          14
Allergies\nNo known drug allergies                                     11
Clinician Leading Ward Round\nDr. Edward Aaron O'Connor (SpR)          10
Allergies\nNone            

In [34]:
section_chunks.loc[
    section_chunks["chunk_text"].str.contains(
        r"#NAME\?",
        na=False,
        regex=True,
    ),
    [
        "clinical_note_id",
        "note_subject",
        "chunk_text",
    ],
].head(20)

,clinical_note_id,note_subject,chunk_text
812,0940fa5e-f355-4cea-8e17-61465ea4ff7c,Physio-led education session,#NAME?
962,661de17e-a4a7-4549-857a-b78e5ce2decf,Dietitian review for post-op nutrition,#NAME?
1682,d8df8be7-3622-4782-857f-8d389ddad694,Dietitian Review,#NAME?
2526,170c8e71-5d7f-4bf7-a74f-41aa2a2d800c,Dietitian review for post-op nutrition,#NAME?
3246,4bf1b7ee-5bdd-40f6-93b3-f576c7158c66,Dietitian Review,#NAME?
4370,a62ff755-7c96-4473-ae8b-f7b468cfb771,Dietary Assessment and Plan,#NAME?
4834,5f6eacde-a2b9-49dd-ac1d-27ac58b0c2de,Dietary review post-op,#NAME?


In [35]:
name_error_ids = section_chunks.loc[
    section_chunks["chunk_text"].str.strip().eq("#NAME?"),
    "clinical_note_id"
].unique()

notes.loc[
    notes["clinical_note_id"].isin(name_error_ids),
    [
        "clinical_note_id",
        "note_subject",
        "clean_note_text",
    ]
]

,clinical_note_id,note_subject,clean_note_text
244,0940fa5e-f355-4cea-8e17-61465ea4ff7c,Physio-led education session,#NAME?
293,661de17e-a4a7-4549-857a-b78e5ce2decf,Dietitian review for post-op nutrition,#NAME?
520,d8df8be7-3622-4782-857f-8d389ddad694,Dietitian Review,#NAME?
787,170c8e71-5d7f-4bf7-a74f-41aa2a2d800c,Dietitian review for post-op nutrition,#NAME?
1014,4bf1b7ee-5bdd-40f6-93b3-f576c7158c66,Dietitian Review,#NAME?
1363,a62ff755-7c96-4473-ae8b-f7b468cfb771,Dietary Assessment and Plan,#NAME?
1498,5f6eacde-a2b9-49dd-ac1d-27ac58b0c2de,Dietary review post-op,#NAME?


In [36]:
INVALID_NOTE_VALUES = {"#NAME?"}

invalid_mask = (
    notes["clean_note_text"]
    .str.strip()
    .isin(INVALID_NOTE_VALUES)
)

print("Invalid notes removed:", invalid_mask.sum())

notes_clean = notes.loc[~invalid_mask].copy()

print("Notes before cleaning:", len(notes))
print("Notes after cleaning:", len(notes_clean))

Invalid notes removed: 7
Notes before cleaning: 1602
Notes after cleaning: 1595


In [37]:
whole_chunks = create_whole_note_chunks(notes_clean)

fixed_chunks = create_fixed_word_chunks(
    notes_clean,
    chunk_size=FIXED_CHUNK_WORDS,
    overlap=FIXED_CHUNK_OVERLAP,
)

section_chunks = create_section_chunks(notes_clean)

## Embedding Models

We compare three embedding approaches:

1. General-purpose: GeminiAI text-embedding-3-small
2. Retrieval-focused: BGE-M3
3. Biomedical retrieval: MedCPT

Each embedding model will be evaluated with the same three chunking strategies:
- Whole-note
- Fixed-size overlapping chunks
- Section-aware chunks

### 1. GeminiAI — text-embedding-3-small

In [4]:
%pip install -q openai


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
%pip install -q python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from google import genai

PROJECT_ROOT = Path.cwd().parent
ENV_PATH = PROJECT_ROOT / ".env"

load_dotenv(ENV_PATH, override=True)

gemini_api_key = os.getenv("GEMINI_API_KEY")

print("Gemini key loaded:", bool(gemini_api_key))

client = genai.Client(api_key=gemini_api_key)

Gemini key loaded: True


In [7]:
# Sanity test: generate one Gemini embedding

test_text = "Patient developed acute confusion following a minor fall."

response = client.models.embed_content(
    model="gemini-embedding-001",
    contents=test_text,
)

test_embedding = response.embeddings[0].values

print("Embedding dimensions:", len(test_embedding))
print("First 10 values:", test_embedding[:10])

Embedding dimensions: 3072
First 10 values: [0.0114165805, -0.014645216, 0.005921604, -0.06154907, 0.00182117, -0.0041579907, -0.014994828, 0.0123460945, 0.0035066789, 0.001972333]


In [8]:
%pip install -q sentence-transformers


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
%pip install nbqa ruff isort


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
import os
from pathlib import Path

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent
ENV_PATH = PROJECT_ROOT / ".env"

load_dotenv(ENV_PATH, override=True)

hf_token = os.getenv("HF_TOKEN")

print("HF token loaded:", bool(hf_token))

HF token loaded: True


In [11]:
from huggingface_hub import login

login(token=hf_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [12]:
print("HF token loaded:", bool(hf_token))

HF token loaded: True


In [13]:
login(token=hf_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:

PROJECT_ROOT = Path.cwd().parent
ENV_PATH = PROJECT_ROOT / ".env"

print("ENV path:", ENV_PATH)
print("ENV exists:", ENV_PATH.exists())

load_dotenv(ENV_PATH, override=True)

hf_token = os.getenv("HF_TOKEN")

print("HF token loaded:", bool(hf_token))

ENV path: /Users/pallavi_chandanshive/projects/clinical-summarization-eval/.env
ENV exists: True
HF token loaded: True


In [15]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

print(PROJECT_ROOT)

/Users/pallavi_chandanshive/projects/clinical-summarization-eval


In [16]:
from sentence_transformers import SentenceTransformer

BGE_MODEL_PATH = PROJECT_ROOT / "models" / "bge-base-en-v1.5"

bge_model = SentenceTransformer(str(BGE_MODEL_PATH))

test_text = "Patient developed acute confusion following a minor fall."

bge_embedding = bge_model.encode(test_text)

print("Embedding dimensions:", len(bge_embedding))
print("First 10 values:", bge_embedding[:10])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding dimensions: 768
First 10 values: [-0.00121067 -0.01221776  0.00434872  0.00537568  0.02330717 -0.00891011
  0.05173944  0.01838303 -0.02286186 -0.01189964]


### 3. MedCPT — Biomedical retrieval embedding

In [17]:
%pip install -q transformers torch


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:

MEDCPT_QUERY_PATH = PROJECT_ROOT / "models" / "MedCPT-Query-Encoder"
MEDCPT_ARTICLE_PATH = PROJECT_ROOT / "models" / "MedCPT-Article-Encoder"

print("Query model exists:", MEDCPT_QUERY_PATH.exists())
print("Article model exists:", MEDCPT_ARTICLE_PATH.exists())

Query model exists: True
Article model exists: True


In [23]:
query_tokenizer = AutoTokenizer.from_pretrained(
    MEDCPT_QUERY_PATH,
    local_files_only=True
)

query_model = AutoModel.from_pretrained(
    MEDCPT_QUERY_PATH,
    local_files_only=True
)

article_tokenizer = AutoTokenizer.from_pretrained(
    MEDCPT_ARTICLE_PATH,
    local_files_only=True
)

article_model = AutoModel.from_pretrained(
    MEDCPT_ARTICLE_PATH,
    local_files_only=True
)

query_model.eval()
article_model.eval()

print("MedCPT models loaded successfully")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MedCPT models loaded successfully


In [24]:
test_query = "What caused the patient's acute confusion?"

test_article = """
Patient developed acute confusion following a minor fall.
CT head showed a subdural hygroma. Mild hyponatremia and dehydration
were also documented.
"""

In [25]:
with torch.no_grad():

    query_inputs = query_tokenizer(
        test_query,
        return_tensors="pt",
        truncation=True,
        padding=True,
    )

    query_output = query_model(**query_inputs)
    query_embedding = query_output.last_hidden_state[:, 0, :]


    article_inputs = article_tokenizer(
        test_article,
        return_tensors="pt",
        truncation=True,
        padding=True,
    )

    article_output = article_model(**article_inputs)
    article_embedding = article_output.last_hidden_state[:, 0, :]

In [26]:
print("Query embedding shape:", query_embedding.shape)
print("Article embedding shape:", article_embedding.shape)

print("First 10 query values:")
print(query_embedding[0][:10])

print("\nFirst 10 article values:")
print(article_embedding[0][:10])

Query embedding shape: torch.Size([1, 768])
Article embedding shape: torch.Size([1, 768])
First 10 query values:
tensor([ 0.0749, -0.0908, -0.0609, -0.3313, -0.0904, -0.0067, -0.3357, -0.0689,
        -0.1430, -0.2947])

First 10 article values:
tensor([-0.3596, -0.0933, -0.0052, -0.3221, -0.1385, -0.1546, -0.6965, -0.4492,
        -0.0412,  0.0531])
